In [1]:
import requests
import pandas as pd
from sklearn.model_selection import train_test_split

In [2]:
class RussianOllamaClient:
    def __init__(self, model_name="llama3:latest"):
        self.model_name = model_name
        self.base_url = "http://localhost:11434"
        print(f"Используем модель: {model_name}")
    
    def make_request(self, prompt, temperature):
        data = {
            "model": self.model_name,
            "prompt": prompt,
            "stream": False,
            "options": {
                "temperature": temperature,
                "num_predict": 100
            }
        }
        
        try:
            response = requests.post(
                f"{self.base_url}/api/generate", 
                json=data, 
                timeout=500
            )
            
            if response.status_code == 200:
                result = response.json()
                return result['response']
            else:
                print(f"❌ Ошибка {response.status_code}")
                return ""
                
        except Exception as e:
            print(f"❌ Исключение: {e}")
            return ""

In [3]:
client = RussianOllamaClient("llama3:8b")  
    
test_response = client.make_request("Скажи 'тест пройден'", temperature=0.3)
if test_response:
    print("✅ Модель работает")
else:
    print("❌ Модель не отвечает")

Используем модель: llama3:8b
✅ Модель работает


In [4]:
try:
    df = pd.read_csv('ru_cefr_short.csv')
    print("✅ Файл найден:")
except:
    print("❌ Файл не найден, используем тестовые данные:")
    test_data = {
        'fragment': [
            "Весной, летом и осенью почти каждую субботу он играет в футбол. У них в школе есть футбольная команда. А зимой он играет в хоккей. Ещё мы любим театр.",
            "Вчера я весь вечер повторял грамматику, учил слова. В контрольной работе я сделал 3 ошибки. Питер и Кен написали всё без ошибок. Преподаватель сказал, что они делают большие успехи.",
            "В такой ситуации особенно сложно работающим студентам, которые должны находить время и для учёбы, и для работы. Это нередко вызывает стресс.",
            "Как редкое животное они стоили очень дорого и являлись предметом роскоши. Археологи нашли останки этих животных в развалинах домов богатых людей четвёртого века нашей эры.",
            "Он увлёкся энтомологией ещё мальчиком и в детстве познакомился с основными учёными трудами в этой области.",
            "Так же радиация — это частица, которая летит на огромной скорости. Сама частица может быть почти любой: от атомного ядра до электрона или фотона."
        ],
        'textbook-assigned cefr level': [1, 2, 3, 4, 5, 6]
    }
    df = pd.DataFrame(test_data)

df

✅ Файл найден:


,fragment,textbook-assigned cefr level
0,"Весной, летом и осенью почти каждую субботу он...",1
1,"Все говорят, что мама хорошая хозяйка. А ещё н...",1
2,На каждой двери красные плакаты и красные фона...,1
3,"Я считаю деньги, в час обедаю в кафе, а потом ...",1
4,Магазин «Чёрный квадрат» открывается в 9 часов...,1
...,...,...
7317,Утечка мозгов стала ключевым трендом междунаро...,6
7318,"По оценкам менеджеров «Промы», такая ситуация ...",6
7319,"Но это не мы, а техно-мемы заполоняют мир благ...",6
7320,Mapillary использует программное обеспечение д...,6


In [5]:
train_texts, val_texts, train_labels, val_labels = train_test_split(
    df['fragment'].values,
    df['textbook-assigned cefr level'].values,
    test_size=0.2,
    random_state=42,
    stratify=df['textbook-assigned cefr level']
)

len(train_texts)

5857

# Аугментация В2

In [6]:
texts_to_augment_b2 = []

for i in range(len(train_labels)):
  if train_labels[i] == 4:
    texts_to_augment_b2.append(train_texts[i])

texts_to_augment_b2 = texts_to_augment_b2[:120]
len(texts_to_augment_b2)

120

In [7]:
def create_prompt(text):
    return f'''Перепиши данный текст на русском языке так, чтобы его уровень сложности соответствовал С2, сохранив исходный смысл, ключевую информацию и общую тему.

Требования к результату:
- Новый текст должен быть написан на русском языке.
- Уровень сложности должен соответствовать С2.
- Не упрощай текст.
- Сохрани основной смысл исходного текста.
- Не меняй тему текста.
- Не добавляй новые факты, которых нет в исходном тексте.
- Итоговый текст должен звучать естественно и связно.
- Длина нового текста может быть больше исходного, если это необходимо для достижения уровня С2.

Требования к уровню С2:
- Владение сложной лексикой, в том числе научного стиля речи, устойчивыми оборотами речи, идиомами. 
- Синтаксические модели выражения логической связи между объектами мысли, употребление вводных слов, конструкций научного стиля речи: что является чем, представляет собой, относится к, предназначено для, обозначается как, ведет к чему, состоит из чего, из чего следует что, характеризуется чем, из чего можно сделать вывод и пр. 
- Владение языком на профессиональном уровне, включающем преподавание русского языка как иностранного и проведение научно-исследовательской работы. 
- Очень сложная структура предложения с множеством связей, сложные союзы и предлоги, устойчивые выражения, связь через двоеточие, причастные и деепричастные обороты, перечисления, использование прецедентных текстов.

Исходный текст: "{text}"

Верни ТОЛЬКО один новый текст на русском языке без пояснений, комментариев и кавычек:'''

In [8]:
def augment_b2(temperature):
    df_pred = pd.DataFrame(columns=['text', 'augmented-text'])

    n = 1
    print(f"Аугментируем {len(texts_to_augment_b2)} текстов...")
    
    for text in texts_to_augment_b2:
    
        prompt = create_prompt(text)
        augmented_text = client.make_request(prompt, temperature)
        
        print(f"\n{n}. Реальный текст:\n\t{text}")
        print(f"Аугметированный текст:\n\t{augmented_text}")
       
        new_row = pd.DataFrame({
            'text': [text],
            'augmented-text': [augmented_text]
        })
        df_pred = pd.concat([df_pred, new_row], ignore_index=True)
    
        n+=1
    
    df_pred.to_csv(f"c2_from_b2_augmented_llama3_temp_{str(temperature).replace('.', '_')}.csv")

## Temperature = 0.0

In [9]:
temperature = 0.0
augment_b2(temperature)

Аугментируем 120 текстов...

1. Реальный текст:
	По дороге к её дому его ожидали разные препятствия: друзья и подруги невесты не хотели его пропускать, и он должен был что-нибудь подарить или заплатить им.
Аугметированный текст:
	На пути к ее дому его ожидали различные барьеры: друзья и подруги невесты не желали позволить ему проходить мимо, и он был вынужден предпринять какое-то действие или предоставить им компенсацию.

2. Реальный текст:
	Что же выявил наш анализ? Самое главное — длительность жизни людей творческих профессий была значительно больше, чем средняя, во все эпохи.
Аугметированный текст:
	Анализ нашего исследования привел к следующим результатам. Важнейшим из них является обнаружение значительной разницы в продолжительности жизни людей, занимающихся творческими профессиями, по сравнению с общей средней продолжительностью жизни во всех эпохах.

3. Реальный текст:
	В начале конференции немецкий социолог сделал небольшое сообщение о проблемах экологического самосознания в ФР

## Temperature = 0.1

In [10]:
temperature = 0.1
augment_b2(temperature)

Аугментируем 120 текстов...
The history saving thread hit an unexpected error (OperationalError('attempt to write a readonly database')).History will not be written to the database.

1. Реальный текст:
	По дороге к её дому его ожидали разные препятствия: друзья и подруги невесты не хотели его пропускать, и он должен был что-нибудь подарить или заплатить им.
Аугметированный текст:
	На пути к ее дому он столкнулся с рядом препятствий: друзья и подруги невесты не желали его пропускать, и он был вынужден предпринять какое-то действие или компенсировать их ожидания.

2. Реальный текст:
	Что же выявил наш анализ? Самое главное — длительность жизни людей творческих профессий была значительно больше, чем средняя, во все эпохи.
Аугметированный текст:
	Анализ нашего исследования привел к следующим результатам. В частности, выяснилось, что длительно карьерные пути людей из творческих профессий отличались значительной продолжительностью жизни по сравнению с общей тенденцией в различных эпохах.

3.

## Temperature = 0.2

In [11]:
temperature = 0.2
augment_b2(temperature)

Аугментируем 120 текстов...

1. Реальный текст:
	По дороге к её дому его ожидали разные препятствия: друзья и подруги невесты не хотели его пропускать, и он должен был что-нибудь подарить или заплатить им.
Аугметированный текст:
	На пути к ее доме он предстояло преодолеть различные препятствия: друзья и подруги невесты не желали его пропускать, и он должен был предоставить какое-либо компенсацию или плату за проезд.

2. Реальный текст:
	Что же выявил наш анализ? Самое главное — длительность жизни людей творческих профессий была значительно больше, чем средняя, во все эпохи.
Аугметированный текст:
	Анализ нашего исследования привел к следующим результатам. Важнейшим из них является обнаружение значительной разницы в продолжительности жизни людей, занимающихся творческими профессиями, по сравнению с общей тенденцией для всех эпох.

3. Реальный текст:
	В начале конференции немецкий социолог сделал небольшое сообщение о проблемах экологического самосознания в ФРГ, что вызвало бурную дискус

## Temperature = 0.3

In [12]:
temperature = 0.3
augment_b2(temperature)

Аугментируем 120 текстов...

1. Реальный текст:
	По дороге к её дому его ожидали разные препятствия: друзья и подруги невесты не хотели его пропускать, и он должен был что-нибудь подарить или заплатить им.
Аугметированный текст:
	На пути к ее доме он встречал различные барьеры: друзья и подруги невесты не желали позволить ему проходить мимо, и он был вынужден предпринять какое-то действие или уплатить им за это.

2. Реальный текст:
	Что же выявил наш анализ? Самое главное — длительность жизни людей творческих профессий была значительно больше, чем средняя, во все эпохи.
Аугметированный текст:
	Анализ наших данных привел к следующему результату: наиболее существенное различие между продолжительностью жизни людей, занимающихся творческой деятельностью, и средней продолжительностью жизни в целом заключается в том, что первая была значительно более длительной, чем вторая, на протяжении всех эпох.

3. Реальный текст:
	В начале конференции немецкий социолог сделал небольшое сообщение о пробл

## Temperature = 0.4

In [13]:
temperature = 0.4
augment_b2(temperature)

Аугментируем 120 текстов...

1. Реальный текст:
	По дороге к её дому его ожидали разные препятствия: друзья и подруги невесты не хотели его пропускать, и он должен был что-нибудь подарить или заплатить им.
Аугметированный текст:
	На пути к ее дому его ожидали различные барьеры: друзья и подруги невесты не желали позволить ему пройти мимо, и он был вынужден предпринять какое-то действие или предоставить компенсацию.

2. Реальный текст:
	Что же выявил наш анализ? Самое главное — длительность жизни людей творческих профессий была значительно больше, чем средняя, во все эпохи.
Аугметированный текст:
	Анализ нашего исследования привел к следующему результату: наиболее значимым выводом является существенное отличие продолжительности жизни людей, занимающихся творческими профессиями, от средней продолжительности жизни в различных эпохах.

3. Реальный текст:
	В начале конференции немецкий социолог сделал небольшое сообщение о проблемах экологического самосознания в ФРГ, что вызвало бурную диск

## Temperature = 0.5

In [14]:
temperature = 0.5
augment_b2(temperature)

Аугментируем 120 текстов...

1. Реальный текст:
	По дороге к её дому его ожидали разные препятствия: друзья и подруги невесты не хотели его пропускать, и он должен был что-нибудь подарить или заплатить им.
Аугметированный текст:
	Путь к ее дому представлял собой серию препятствий: коллеги невесты не были готовы его пропустить, и он должен был предложить какое-то вознаграждение или оплатить их услуги.

2. Реальный текст:
	Что же выявил наш анализ? Самое главное — длительность жизни людей творческих профессий была значительно больше, чем средняя, во все эпохи.
Аугметированный текст:
	Анализ нашей исследовательской группы привел к следующим результатам. Важнейшим из них является обнаружение существенного различия в продолжительности жизни людей, занимающихся творческими профессиями, по сравнению с общим уровнем ожидаемой продолжительности жизни для других групп населения. Это различие наблюдалось в различных эпохах истории и не зависит от времени или культуры.

3. Реальный текст:
	В начал

## Temperature = 0.6

In [15]:
temperature = 0.6
augment_b2(temperature)

Аугментируем 120 текстов...

1. Реальный текст:
	По дороге к её дому его ожидали разные препятствия: друзья и подруги невесты не хотели его пропускать, и он должен был что-нибудь подарить или заплатить им.
Аугметированный текст:
	На пути к ее доме его ожидали различные помехи: друзья и подруги невесты не желали позволить ему проходить мимо, и он должен был предоставить какое-либо вознаграждение или выполнить соответствующие выплаты.

2. Реальный текст:
	Что же выявил наш анализ? Самое главное — длительность жизни людей творческих профессий была значительно больше, чем средняя, во все эпохи.
Аугметированный текст:
	Анализ подтвердил ожидания: длительность жизни людей, занимающихся творческими профессиями, была существенно больше средней продолжительности жизни в всех эпохах. Это наблюдение не является исключением, аrather, это подтверждает общую тенденцию, которая прослеживается на протяжении всей истории человечества. Важно подчеркнуть, что подобное исследование может помочь лучше поня

## Temperature = 0.7

In [16]:
temperature = 0.7
augment_b2(temperature)

Аугментируем 120 текстов...

1. Реальный текст:
	По дороге к её дому его ожидали разные препятствия: друзья и подруги невесты не хотели его пропускать, и он должен был что-нибудь подарить или заплатить им.
Аугметированный текст:
	При подходе к ее доме ожидает ряд препятствий: друзья и подруги невесты не намерены позволить ему проехать, и он вынужден предлагать компенсацию или оплату за пропуск. В этом случае, чтобы преодолеть возникшие барьеры, его дожидается необходимость предложить что-либо в виде подарка или вознаграждения для друзей и подруг

2. Реальный текст:
	Что же выявил наш анализ? Самое главное — длительность жизни людей творческих профессий была значительно больше, чем средняя, во все эпохи.
Аугметированный текст:
	Анализ нашего исследования позволил нам установить, что длительность жизни людей, занимающихся творческими профессиями, была существенно больше средней продолжительности жизни в различных эпохах. В частности, наша работа выявляет обширные различия между продолжит

## Temperature = 0.8

In [17]:
temperature = 0.8
augment_b2(temperature)

Аугментируем 120 текстов...

1. Реальный текст:
	По дороге к её дому его ожидали разные препятствия: друзья и подруги невесты не хотели его пропускать, и он должен был что-нибудь подарить или заплатить им.
Аугметированный текст:
	В путь к ее резиденции лежали различные барьеры: знакомые и приятели невесты не собирались его пропускать, и он должен был найти способ устроить их или заплатить за проезд.

2. Реальный текст:
	Что же выявил наш анализ? Самое главное — длительность жизни людей творческих профессий была значительно больше, чем средняя, во все эпохи.
Аугметированный текст:
	Анализ выяснил следующее. Центральный результат нашего исследования заключается в том, что продолжительность жизни людей, практиковших творческие профессии, была существенно более долгой, чем средняя для соответствующих эпох.

3. Реальный текст:
	В начале конференции немецкий социолог сделал небольшое сообщение о проблемах экологического самосознания в ФРГ, что вызвало бурную дискуссию.
Аугметированный текст:

## Temperature = 0.9

In [18]:
temperature = 0.9
augment_b2(temperature)

Аугментируем 120 текстов...

1. Реальный текст:
	По дороге к её дому его ожидали разные препятствия: друзья и подруги невесты не хотели его пропускать, и он должен был что-нибудь подарить или заплатить им.
Аугметированный текст:
	На пути к ее домому его ожидает целый ряд препятствий: друзья и подруги невесты не намерены уступить ему проход, требуя от него какой-либо компенсации или дара в виде.

2. Реальный текст:
	Что же выявил наш анализ? Самое главное — длительность жизни людей творческих профессий была значительно больше, чем средняя, во все эпохи.
Аугметированный текст:
	Анализ нашей команды обнаружил существенное отличие в продолжительности жизни людей, занимающихся творческими профессиями. Длительность их жизни была существенно больше средней, как в эпохи прошлого, так и в наши дни. Всякий раз, когда мы изучали различные периоды истории, результат был одинаков: люди, которые посвятили свою жизнь созданию искусства, научным исследованиям или философским

3. Реальный текст:
	В нач

## Temperature = 1.0

In [19]:
temperature = 1.0
augment_b2(temperature)

Аугментируем 120 текстов...

1. Реальный текст:
	По дороге к её дому его ожидали разные препятствия: друзья и подруги невесты не хотели его пропускать, и он должен был что-нибудь подарить или заплатить им.
Аугметированный текст:
	На пути к ее доме существовала целая серия препятствий: представители социальной сети невесты не стремились позволить ему пройти мимо, и он был вынужден предложить что-то в качестве подарка или компенсации за их услуги.

2. Реальный текст:
	Что же выявил наш анализ? Самое главное — длительность жизни людей творческих профессий была значительно больше, чем средняя, во все эпохи.
Аугметированный текст:
	Результаты нашего анализа выявили существенное отличие длительности жизни людей, занимающихся творческими профессиями, от среднего показателя, что является обобщим явлением в различных эпохах. 

Обратите внимание, что я оставил все оригинальные идеи и фрагменты текста, а также сохранил стиль и язык, соответствующие уровню С2.

3. Реальный текст:
	В начале конфере

## Temperature = 1.1

In [20]:
temperature = 1.1
augment_b2(temperature)

Аугментируем 120 текстов...

1. Реальный текст:
	По дороге к её дому его ожидали разные препятствия: друзья и подруги невесты не хотели его пропускать, и он должен был что-нибудь подарить или заплатить им.
Аугметированный текст:
	Несколько помех ожидали его на пути к доме невесты: друзья и подруги той не хотели дать ему проезд, требуя от него какого-либо символического вознаграждения или платы за пуск.

2. Реальный текст:
	Что же выявил наш анализ? Самое главное — длительность жизни людей творческих профессий была значительно больше, чем средняя, во все эпохи.
Аугметированный текст:
	Наш анализ продемонстрировал существенное отличие между продолжительностью жизни людей, занимающихся творческими профессиями, и средней продолжительностью жизни в различных эпохах. Прямое обнаружение является заключением о том, что длительность жизни представителей этих групп была значительно больше, чем в остальных коллективах населения.

3. Реальный текст:
	В начале конференции немецкий социолог сделал н

## Temperature = 1.2

In [21]:
temperature = 1.2
augment_b2(temperature)

Аугментируем 120 текстов...

1. Реальный текст:
	По дороге к её дому его ожидали разные препятствия: друзья и подруги невесты не хотели его пропускать, и он должен был что-нибудь подарить или заплатить им.
Аугметированный текст:
	В пути к ее domicile ожидался спектр помех: друзья и приятельницы невесты не желали пропустить его, и он должен был предложить quelque chose или выставить счет им.

2. Реальный текст:
	Что же выявил наш анализ? Самое главное — длительность жизни людей творческих профессий была значительно больше, чем средняя, во все эпохи.
Аугметированный текст:
	Анализ нашего исследования вывел наиболее значимое открытие - люди, занимающиеся творческими профессиями, проживали намного дольше, чем общепринятый средний уровень, во всех эпохах истории.

3. Реальный текст:
	В начале конференции немецкий социолог сделал небольшое сообщение о проблемах экологического самосознания в ФРГ, что вызвало бурную дискуссию.
Аугметированный текст:
	Находящаяся на старте конференции немецкий 

## Temperature = 1.3

In [22]:
temperature = 1.3
augment_b2(temperature)

Аугментируем 120 текстов...

1. Реальный текст:
	По дороге к её дому его ожидали разные препятствия: друзья и подруги невесты не хотели его пропускать, и он должен был что-нибудь подарить или заплатить им.
Аугметированный текст:
	Дорожная сеть к ее жилищу была усеяна разными барьерами: друзья и подружки невесты не желали пустить его без обмена дарами или платежами.

2. Реальный текст:
	Что же выявил наш анализ? Самое главное — длительность жизни людей творческих профессий была значительно больше, чем средняя, во все эпохи.
Аугметированный текст:
	Анализ наше привел к далеко неожиданному выводу. Оказалось, что длитию жизни людей из творческих профессий в любую эпоху была значительно выше среднего уровня. Это обстоятельство свидетельствует о том, что занятость творческой деятельностью имеет свой уникальный эффект на процессaging, который не может быть учетен при анализе средних значений для всей популяции

3. Реальный текст:
	В начале конференции немецкий социолог сделал небольшое сообще

## Temperature = 1.4

In [23]:
temperature = 1.4
augment_b2(temperature)

Аугментируем 120 текстов...

1. Реальный текст:
	По дороге к её дому его ожидали разные препятствия: друзья и подруги невесты не хотели его пропускать, и он должен был что-нибудь подарить или заплатить им.
Аугметированный текст:
	Куда дойдет путь жениха к ее домофикацией, он может столкнуться с разнообразными помехами: круг знакомых невесты не стремится его пропускать, а для этого он должен выполнить какое-либо благородное действие или заплатить им за путь.

2. Реальный текст:
	Что же выявил наш анализ? Самое главное — длительность жизни людей творческих профессий была значительно больше, чем средняя, во все эпохи.
Аугметированный текст:
	Анализ выявил следующие результаты. Один из наиболее важных факторов - это длительность жизни людей, занимающихся творческими профессиями, которая значительно превосходила среднюю долголетительность во всех эпохах. В этом отношении наш анализ подтвердил устойчивую связь между творческим потенциалом и продолжительностью жизни, подчеркивающую важность ф

## Temperature = 1.5

In [24]:
temperature = 1.5
augment_b2(temperature)

Аугментируем 120 текстов...

1. Реальный текст:
	По дороге к её дому его ожидали разные препятствия: друзья и подруги невесты не хотели его пропускать, и он должен был что-нибудь подарить или заплатить им.
Аугметированный текст:
	Путь к ее домошество препятствиями разнорахой сопровождался: союзные круги невесты и ее друзья пытались предотвратить его проходимость, поскольку он должен был предлагать компромисс или уплачивать за разрешение.

2. Реальный текст:
	Что же выявил наш анализ? Самое главное — длительность жизни людей творческих профессий была значительно больше, чем средняя, во все эпохи.
Аугметированный текст:
	Анализ показал следующее: динамическая продолжительность жизни людей творческих профессий была существенно выше среднего уровня в любой эпохе, что является важным выводом для понимания факторов, которые влияют на карьеру и общую производительность индивидуумов.

3. Реальный текст:
	В начале конференции немецкий социолог сделал небольшое сообщение о проблемах экологическо

# Аугментация С1

In [25]:
texts_to_augment_c1 = []

for i in range(len(train_labels)):
  if train_labels[i] == 5:
    texts_to_augment_c1.append(train_texts[i])

texts_to_augment_c1 = texts_to_augment_c1[:120]
len(texts_to_augment_c1)

120

In [26]:
def create_prompt(text):
    return f'''Перепиши данный текст на русском языке так, чтобы его уровень сложности соответствовал С2, сохранив исходный смысл, ключевую информацию и общую тему.

Требования к результату:
- Новый текст должен быть написан на русском языке.
- Уровень сложности должен соответствовать С2.
- Не упрощай текст.
- Сохрани основной смысл исходного текста.
- Не меняй тему текста.
- Не добавляй новые факты, которых нет в исходном тексте.
- Итоговый текст должен звучать естественно и связно.
- Длина нового текста может быть больше исходного, если это необходимо для достижения уровня С2.

Требования к уровню С2:
- Владение сложной лексикой, в том числе научного стиля речи, устойчивыми оборотами речи, идиомами. 
- Синтаксические модели выражения логической связи между объектами мысли, употребление вводных слов, конструкций научного стиля речи: что является чем, представляет собой, относится к, предназначено для, обозначается как, ведет к чему, состоит из чего, из чего следует что, характеризуется чем, из чего можно сделать вывод и пр. 
- Владение языком на профессиональном уровне, включающем преподавание русского языка как иностранного и проведение научно-исследовательской работы. 
- Очень сложная структура предложения с множеством связей, сложные союзы и предлоги, устойчивые выражения, связь через двоеточие, причастные и деепричастные обороты, перечисления, использование прецедентных текстов.

Исходный текст: "{text}"

Верни ТОЛЬКО один новый текст на русском языке без пояснений, комментариев и кавычек:'''

In [27]:
def augment_c1(temperature):
    df_pred = pd.DataFrame(columns=['text', 'augmented-text'])

    n = 1
    print(f"Аугментируем {len(texts_to_augment_c1)} текстов...")
    
    for text in texts_to_augment_c1:
    
        prompt = create_prompt(text)
        augmented_text = client.make_request(prompt, temperature)
        
        print(f"\n{n}. Реальный текст:\n\t{text}")
        print(f"Аугметированный текст:\n\t{augmented_text}")
       
        new_row = pd.DataFrame({
            'text': [text],
            'augmented-text': [augmented_text]
        })
        df_pred = pd.concat([df_pred, new_row], ignore_index=True)
    
        n+=1
    
    df_pred.to_csv(f"c2_from_c1_augmented_llama3_temp_{str(temperature).replace('.', '_')}.csv")

## Temperature = 0.0

In [28]:
temperature = 0.0
augment_c1(temperature)

Аугментируем 120 текстов...

1. Реальный текст:
	И я понял: это моя жена. На бледном лице сияют глаза. Большие, выпуклые, чёрные! Это мои глаза, моя душа.
Аугметированный текст:
	Я пришел к выводу, что это моя жена. На ее лице сияют глаза, которые напоминают мои собственные - большие, выпуклые и чёрные. Это не только физическое сходство, но и глубокое понимание, которое связывает нас на уровне души.

В этом тексте я использовал сложную лексику, устойчивые обороты речи и синтаксические модели, характер

2. Реальный текст:
	Получается безвыходная ситуация: человек, обладающий свободным временем и возможностью полностью поменять свою жизнь, замыкается в себе.
Аугметированный текст:
	Ситуация, когда человек, располагающий свободным временем и возможностью кардинально изменить свой жизненный путь, оказывается в ловушке собственных ограничений.

3. Реальный текст:
	Почитание святого Георгия, возникшее, вероятно, на территории Каппадокии в V—VI веках, к IX-XI векам получило распространение по

## Temperature = 0.1

In [29]:
temperature = 0.1
augment_c1(temperature)

Аугментируем 120 текстов...

1. Реальный текст:
	И я понял: это моя жена. На бледном лице сияют глаза. Большие, выпуклые, чёрные! Это мои глаза, моя душа.
Аугметированный текст:
	Я пришел к выводу, что это моя жена. На ее лице сияют глаза, которые напоминают мои собственные - большие, выпуклые и чёрные. Это не только физическое сходство, но и глубокое понимание, которое связывает нас на уровне души.

В этом тексте я использовал сложную лексику, устойчивые обороты речи и синтаксические модели, характер

2. Реальный текст:
	Получается безвыходная ситуация: человек, обладающий свободным временем и возможностью полностью поменять свою жизнь, замыкается в себе.
Аугметированный текст:
	Ситуация, когда человек, располагающий свободным временем и возможностью кардинально изменить свой жизненный путь, оказывается в ловушке собственных ограничений.

3. Реальный текст:
	Почитание святого Георгия, возникшее, вероятно, на территории Каппадокии в V—VI веках, к IX-XI векам получило распространение по

## Temperature = 0.2

In [30]:
temperature = 0.2
augment_c1(temperature)

Аугментируем 120 текстов...

1. Реальный текст:
	И я понял: это моя жена. На бледном лице сияют глаза. Большие, выпуклые, чёрные! Это мои глаза, моя душа.
Аугметированный текст:
	Я пришел к выводу, что это моя жена. На ее лице сияют глаза, которые напоминают мои собственные - большие, выпуклые и чёрные. Это не только физическая схожесть, но и глубокая эмоциональная связь, которая объединяет нас. Моя душа, моя внутренняя природа, моя суть - все это заключено в этих глазах, которые сияют

2. Реальный текст:
	Получается безвыходная ситуация: человек, обладающий свободным временем и возможностью полностью поменять свою жизнь, замыкается в себе.
Аугметированный текст:
	Ситуация, когда человек, располагающий свободным временем и возможностью кардинально изменить свой жизненный путь, оказывается в ловушке собственной изоляции.

3. Реальный текст:
	Почитание святого Георгия, возникшее, вероятно, на территории Каппадокии в V—VI веках, к IX-XI векам получило распространение почти во всех государ

## Temperature = 0.3

In [31]:
temperature = 0.3
augment_c1(temperature)

Аугментируем 120 текстов...

1. Реальный текст:
	И я понял: это моя жена. На бледном лице сияют глаза. Большие, выпуклые, чёрные! Это мои глаза, моя душа.
Аугметированный текст:
	Я пришел к заключению: это моя супруга. На ее бледном лице отражаются мои глаза. Большие, выпуклые, черные - это моя внешняя оболочка, моя душа. В них отражается моя сущность, моя идентичность, которая неотделима от моей любви к ней. Это та самая внутренняя сила, которая направляет мою жизнь и определяет

2. Реальный текст:
	Получается безвыходная ситуация: человек, обладающий свободным временем и возможностью полностью поменять свою жизнь, замыкается в себе.
Аугметированный текст:
	Человеческий выбор, как правило, определяется ограничивающими факторами, которые могут привести к созданию безвыходной ситуации. В частности, когда человек имеет свободное время и возможность кардинально изменить свою жизнь, он может оказаться заключенным в себя, не имея возможности реализовать свои амбиции и стремления.

В этом ко

## Temperature = 0.4

In [32]:
temperature = 0.4
augment_c1(temperature)

Аугментируем 120 текстов...

1. Реальный текст:
	И я понял: это моя жена. На бледном лице сияют глаза. Большие, выпуклые, чёрные! Это мои глаза, моя душа.
Аугметированный текст:
	Я пришел к выводу, что это моя жена. На ее лице бледное освещение, которое является моими глазами, моим духом. Это неотъемлемая часть меня, моя сущность. 

(Примечание: текст написан на уровне С2, он содержит сложную лексику, научный стиль речи, устойчивые обороты речи и синтаксические модели логической связи между объектами мы

2. Реальный текст:
	Получается безвыходная ситуация: человек, обладающий свободным временем и возможностью полностью поменять свою жизнь, замыкается в себе.
Аугметированный текст:
	Ситуация, когда человек, располагающий свободным временем и возможностью кардинально изменить свою жизнь, оказывается в ловушке собственных ограничений.

3. Реальный текст:
	Почитание святого Георгия, возникшее, вероятно, на территории Каппадокии в V—VI веках, к IX-XI векам получило распространение почти во 

## Temperature = 0.5

In [33]:
temperature = 0.5
augment_c1(temperature)

Аугментируем 120 текстов...

1. Реальный текст:
	И я понял: это моя жена. На бледном лице сияют глаза. Большие, выпуклые, чёрные! Это мои глаза, моя душа.
Аугметированный текст:
	Я понял, что это мой внутренний мир. На лице моего сознания сияют глаза моей души. Большие, выпуклые, чёрные - это моя эмоциональная реальность, моя визуальная память о пережитых событиях. В них я вижу себя, свою историю, свои переживания и мысли. Это моя душа, моя индивидуальность, которая является нос

2. Реальный текст:
	Получается безвыходная ситуация: человек, обладающий свободным временем и возможностью полностью поменять свою жизнь, замыкается в себе.
Аугметированный текст:
	Человеческий опыт часто характеризуется парализующим чувством бессилия: индивидуум, обладающий возможностью радикально изменить свой жизненный путь, вместо этого оказывается заключенным в свои собственные ограничения и предубеждения.

3. Реальный текст:
	Почитание святого Георгия, возникшее, вероятно, на территории Каппадокии в V—VI

## Temperature = 0.6

In [34]:
temperature = 0.6
augment_c1(temperature)

Аугментируем 120 текстов...

1. Реальный текст:
	И я понял: это моя жена. На бледном лице сияют глаза. Большие, выпуклые, чёрные! Это мои глаза, моя душа.
Аугметированный текст:
	Я понял, что это мое божественное назначение. На лице, искалеченном страданиями, сияют глаза, которые не только отражают мой внутренний мир, но и символизируют моё самосознание. Большие, выпуклые, чёрные – это мои глаза, моя душа, моя воля, которая направлена на поиск истины.

2. Реальный текст:
	Получается безвыходная ситуация: человек, обладающий свободным временем и возможностью полностью поменять свою жизнь, замыкается в себе.
Аугметированный текст:
	Ситуация, когда индивидуум, располагающий свободным временем и возможностью кардинально изменить свой жизненный путь, заканчивает свою эволюцию.

3. Реальный текст:
	Почитание святого Георгия, возникшее, вероятно, на территории Каппадокии в V—VI веках, к IX-XI векам получило распространение почти во всех государствах Европы и Ближнего Востока.
Аугметированный 

## Temperature = 0.7

In [35]:
temperature = 0.7
augment_c1(temperature)

Аугментируем 120 текстов...

1. Реальный текст:
	И я понял: это моя жена. На бледном лице сияют глаза. Большие, выпуклые, чёрные! Это мои глаза, моя душа.
Аугметированный текст:
	Я понял, что это моя сущность. На лице, облеченном в серый туман, блестят глаза, которые представляют собой зеркало моей души. Большими, выпуклыми и черными они отражают все мои переживания, надежды и страхи, а также мой индивидуальный стиль мышления.

В этом тексте я стремился передать эмоциональное потрясение героя

2. Реальный текст:
	Получается безвыходная ситуация: человек, обладающий свободным временем и возможностью полностью поменять свою жизнь, замыкается в себе.
Аугметированный текст:
	Ситуация, когда индивидуум, располагающий свободным временем и возможностью кардинально изменить свой образ жизни, оказывается в ловушке собственного сознания.

3. Реальный текст:
	Почитание святого Георгия, возникшее, вероятно, на территории Каппадокии в V—VI веках, к IX-XI векам получило распространение почти во всех

## Temperature = 0.8

In [36]:
temperature = 0.8
augment_c1(temperature)

Аугментируем 120 текстов...

1. Реальный текст:
	И я понял: это моя жена. На бледном лице сияют глаза. Большие, выпуклые, чёрные! Это мои глаза, моя душа.
Аугметированный текст:
	Я понял, что это моя сущность. На лице, окрестное бледностью, отражаются мои глаза. Большие, выпуклые, темные! Это мои зрительные органы, моя душевная природа.

В этом тексте я использовал следующую лексику: сущность, окрестное, бледностью, зрительные органы, душевная природа. Синтакс

2. Реальный текст:
	Получается безвыходная ситуация: человек, обладающий свободным временем и возможностью полностью поменять свою жизнь, замыкается в себе.
Аугметированный текст:
	Ситуация вынуждает человека ощущать себя заключенным в собственном мире, лишенным возможности радикально изменить свой жизненный путь.

3. Реальный текст:
	Почитание святого Георгия, возникшее, вероятно, на территории Каппадокии в V—VI веках, к IX-XI векам получило распространение почти во всех государствах Европы и Ближнего Востока.
Аугметированный т

## Temperature = 0.9

In [37]:
temperature = 0.9
augment_c1(temperature)

Аугментируем 120 текстов...

1. Реальный текст:
	И я понял: это моя жена. На бледном лице сияют глаза. Большие, выпуклые, чёрные! Это мои глаза, моя душа.
Аугметированный текст:
	Я пришёл к заключению: это моя суть. На измождённом лице меркнут взгляды. Великолепные, выпуклые, тёмные! Это мои зрительные органы, моя душа, выраженная в этом мгновении.

Обратите внимание, что новый текст соответствует уровню С2 по следующим критериям:

* Используется сложная лексика: "измождённый

2. Реальный текст:
	Получается безвыходная ситуация: человек, обладающий свободным временем и возможностью полностью поменять свою жизнь, замыкается в себе.
Аугметированный текст:
	Человек, обладающий ресурсами для реализации радикальных изменений и имеющий свободу выбора в определении направления своей жизни, часто оказывается в ловушке собственных ограничений и предубеждений. Эта ситуация может стать безвыходной, когда индивидуум отказывается от возможности трансформации, оставаясь наедине с внутренними блоками

## Temperature = 1.0

In [38]:
temperature = 1.0
augment_c1(temperature)

Аугментируем 120 текстов...

1. Реальный текст:
	И я понял: это моя жена. На бледном лице сияют глаза. Большие, выпуклые, чёрные! Это мои глаза, моя душа.
Аугметированный текст:
	Понимание реальности внезапно зарядило меня эмоциональной энергией, и я пришел к заключению: это моя жизнь. Глаза бледного лица сияли передо мной, как два чёрных монолита, выдающиеся из серого фона. Это мои глаза, мой внутренний мир, моя душа, которая рвётся наружу, чтобы достичь совершенства

2. Реальный текст:
	Получается безвыходная ситуация: человек, обладающий свободным временем и возможностью полностью поменять свою жизнь, замыкается в себе.
Аугметированный текст:
	Непреодолимая ситуация складывается: человек, располагающийся в состоянии полной свободой времени и способностью радикально изменить свою жизнь, фиксирует себя в собственной инерции.

3. Реальный текст:
	Почитание святого Георгия, возникшее, вероятно, на территории Каппадокии в V—VI веках, к IX-XI векам получило распространение почти во всех г

## Temperature = 1.1

In [39]:
temperature = 1.1
augment_c1(temperature)

Аугментируем 120 текстов...

1. Реальный текст:
	И я понял: это моя жена. На бледном лице сияют глаза. Большие, выпуклые, чёрные! Это мои глаза, моя душа.
Аугметированный текст:
	Также я понял, что это моя спутница жизни. На измученном лице отражаются мои глаза. Большие, выпуклые, темные - это моя душа, моя суть. В этих чёрных очах я вижу себя, свою историю, свои эмоции, свою индивидуальность. Это моя обитель, моя стоянка в море жизни. И когда я нахожу себя

2. Реальный текст:
	Получается безвыходная ситуация: человек, обладающий свободным временем и возможностью полностью поменять свою жизнь, замыкается в себе.
Аугметированный текст:
	Фатальная ловушка: индивидуум, обладая неиспользованным временем и способностью кардинально преобразовать свою жизнь, отрезает пути для себя. 

Обратите внимание, что я не добавлял новые факты, не упрощал текст, не изменял тему и не менял длину текста. Новый текст на русском языке соответствует уровню С2 по всем требованиям.

3. Реальный текст:
	Почитани

## Temperature = 1.2

In [40]:
temperature = 1.2
augment_c1(temperature)

Аугментируем 120 текстов...

1. Реальный текст:
	И я понял: это моя жена. На бледном лице сияют глаза. Большие, выпуклые, чёрные! Это мои глаза, моя душа.
Аугметированный текст:
	В тот момент я понял: это моя спутница жизни. На бледном лице сияют мои зрительные органы, которые изъятно отличаются от обычных. Именно их форма - большие, выпуклые, темные глаза, которая является индивидуальной характеристикой и моим зеркалом для понимания самой себя.

2. Реальный текст:
	Получается безвыходная ситуация: человек, обладающий свободным временем и возможностью полностью поменять свою жизнь, замыкается в себе.
Аугметированный текст:
	Человек, обладает ли он свободным временем для реализации своих стремлений или не имеющий ограничивающих факторов, оказывается в затруднительной ситуации, когда потенциал личного преобразования остается несбытым. В этом случае его жизнь может быть представлена как изолированным микрокосмом, где он сам себя ограничивает и блокирует собственные возможности развития.



## Temperature = 1.3

In [41]:
temperature = 1.3
augment_c1(temperature)

Аугментируем 120 текстов...

1. Реальный текст:
	И я понял: это моя жена. На бледном лице сияют глаза. Большие, выпуклые, чёрные! Это мои глаза, моя душа.
Аугметированный текст:
	Я пришел к убеждению, что это моя супруга. На воздушном облике ее лица сияют глаза, огромные, выпуклые, тёмные. Это мои глаза, моя душа, которые отражают всю глубину моей души.

(Уровень сложности: С2)

2. Реальный текст:
	Получается безвыходная ситуация: человек, обладающий свободным временем и возможностью полностью поменять свою жизнь, замыкается в себе.
Аугметированный текст:
	Ситуация, которая складывается перед нами, может быть определена как безвыходная. Ведь человек, обладающий свободным временем и возможностью радикально изменить свой жизненный путь, в последние годы все чаще предпочитает самоизолироваться.

В этом контексте стоит обратить внимание на когнитивный фактор – влияние времени на принятие решений. Человек, обладающий свободным временем,

3. Реальный текст:
	Почитание святого Георгия, возник

## Temperature = 1.4

In [42]:
temperature = 1.4
augment_c1(temperature)

Аугментируем 120 текстов...

1. Реальный текст:
	И я понял: это моя жена. На бледном лице сияют глаза. Большие, выпуклые, чёрные! Это мои глаза, моя душа.
Аугметированный текст:
	Я понял истину: это моя супруга. На пастельном оттенке лица светятся зрелища. Велики, выпукли, чёрны! Это мои зрелища, моя душа. 

Истина эта проникла в глубокие ямы моей души, прояснив контуры человеческой природы. Сияние глаз, которое обливает поверхность лице, не столько это физическое изображ

2. Реальный текст:
	Получается безвыходная ситуация: человек, обладающий свободным временем и возможностью полностью поменять свою жизнь, замыкается в себе.
Аугметированный текст:
	Бессилийная спираль: индивидуум с возможностью радикальной переменой жизни создает для себя изоляцию, затрагивая потенциал существенного преображения.

3. Реальный текст:
	Почитание святого Георгия, возникшее, вероятно, на территории Каппадокии в V—VI веках, к IX-XI векам получило распространение почти во всех государствах Европы и Ближнег

## Temperature = 1.5

In [43]:
temperature = 1.5
augment_c1(temperature)

Аугментируем 120 текстов...

1. Реальный текст:
	И я понял: это моя жена. На бледном лице сияют глаза. Большие, выпуклые, чёрные! Это мои глаза, моя душа.
Аугметированный текст:
	Постижение собственного Я заключается в распознавании своей идентичности. В инстинктивном стремлении понимания myself я осознаю важность женских черт в моей жизни. Оба лика сияют похожими глазами, огромными и выпуклыми, как чернота, обладающими магической силой и символизирующими мою душу.

2. Реальный текст:
	Получается безвыходная ситуация: человек, обладающий свободным временем и возможностью полностью поменять свою жизнь, замыкается в себе.
Аугметированный текст:
	Сформировывается затруднительная ситуация: индивидуум, располагающий свободным временем и возможностью кардинально переменить свою жизнь, замкнут на себя. 

Это означает, что человек, обладая такими ресурсами, не использует их для трансформации своей жизни и обретения новых перспектив, вместо этого сковывая себя внутренними барьерами и ограничени